In [ ]:
!pip install crewai crewai-tools pandas pyyaml -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.8/195.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 836.4/836.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

API

In [ ]:
from google.colab import userdata, files
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

Uploading transaction and Yaml files

In [ ]:

os.makedirs("config", exist_ok=True)

uploaded = files.upload()

FILE_PATH = None
for filename, content in uploaded.items():
    if filename.endswith((".yaml", ".yml")):
        dest = os.path.join("config", filename)
    else:
        dest = filename
        FILE_PATH = filename
    with open(dest, "wb") as f:
        f.write(content)

if FILE_PATH is None:
    FILE_PATH = "transactions.csv"

print(f"Using file: {FILE_PATH}")
print("config/ contains:", os.listdir("config"))

Saving agents.yaml to agents.yaml
Saving tasks.yaml to tasks.yaml
Saving transactions.csv to transactions.csv
Using file: transactions.csv
config/ contains: ['tasks.yaml', 'agents.yaml']


Creating the custm tool

In [ ]:
from typing import Type, List
import pandas as pd
from crewai.tools import BaseTool
from pydantic import BaseModel, Field


class TransactionLoaderInput(BaseModel):
    """Input schema for TransactionLoaderTool."""

    file_path: str = Field(
        ...,
        description="Path to the transactions CSV file (columns: date, category, description, amount).",
    )


class TransactionLoaderTool(BaseTool):
    name: str = "Transaction Loader"
    description: str = (
        "Loads a personal transactions CSV and returns spend totals grouped by "
        "month and category, plus the month-over-month percent change per "
        "category, as a readable summary. Use this instead of reading the raw "
        "CSV directly."
    )
    args_schema: Type[BaseModel] = TransactionLoaderInput

    def _run(self, file_path: str) -> str:
        try:
            df = pd.read_csv(file_path)
        except FileNotFoundError:
            return f"Error: could not find a CSV at '{file_path}'."
        except Exception as exc:  # noqa: BLE001
            return f"Error reading '{file_path}': {exc}"

        required_cols = {"date", "category", "description", "amount"}
        missing = required_cols - set(df.columns)
        if missing:
            return f"Error: CSV is missing required columns: {sorted(missing)}"

        df["date"] = pd.to_datetime(df["date"])
        df["month"] = df["date"].dt.strftime("%Y-%m")

        monthly = (
            df.groupby(["category", "month"])["amount"]
            .sum()
            .round(2)
            .reset_index()
            .sort_values(["category", "month"])
        )

        lines = ["Monthly spend by category:", ""]
        for category, group in monthly.groupby("category"):
            group = group.sort_values("month")
            months = group["month"].tolist()
            amounts = group["amount"].tolist()
            row = ", ".join(f"{m}: ${a:,.2f}" for m, a in zip(months, amounts))
            lines.append(f"- {category}: {row}")

            if len(amounts) >= 2 and amounts[0] not in (0, None):
                pct_change = ((amounts[-1] - amounts[0]) / amounts[0]) * 100
                lines.append(
                    f"    change from {months[0]} to {months[-1]}: {pct_change:+.1f}%"
                )

        totals = df.groupby("month")["amount"].sum().round(2)
        lines.append("")
        lines.append("Total spend by month:")
        for month, total in totals.items():
            lines.append(f"- {month}: ${total:,.2f}")

        return "\n".join(lines)

In [ ]:
from crewai import LLM, Agent, Crew, Task

llm = LLM(
    model="openai/openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0.7
)

In [ ]:
from pathlib import Path

import yaml


# structure for the output
class FinanceReport(BaseModel):
    summary: str = Field(..., description="A short summary of overall financial health.")
    top_overspend_category: str = Field(
        ..., description="The single category most responsible for overspending."
    )
    health_status: str = Field(
        ..., description='Overall status: "healthy", "warning", or "critical".'
    )
    recommendations: List[str] = Field(
        ..., description="2-4 specific, actionable recommendations."
    )


with open("config/agents.yaml", "r") as f:
    agents_config = yaml.safe_load(f)

with open("config/tasks.yaml", "r") as f:
    tasks_config = yaml.safe_load(f)

transaction_loader_tool = TransactionLoaderTool()  # <- defined back in Cell 4, used directly here

transaction_categorizer_agent = Agent(
    config=agents_config["transaction_categorizer_agent"],
    tools=[transaction_loader_tool],
    llm=llm,
)

data_analyst_agent = Agent(
    config=agents_config["data_analyst_agent"],
    llm=llm,
)

financial_advisor_agent = Agent(
    config=agents_config["financial_advisor_agent"],
    llm=llm,
)

categorize_transactions = Task(
    config=tasks_config["categorize_transactions"],
    agent=transaction_categorizer_agent,
)

analyze_spending_trends = Task(
    config=tasks_config["analyze_spending_trends"],
    agent=data_analyst_agent,
    context=[categorize_transactions],
)

generate_financial_report = Task(
    config=tasks_config["generate_financial_report"],
    agent=financial_advisor_agent,
    context=[categorize_transactions, analyze_spending_trends],
    output_pydantic=FinanceReport,
)

finance_crew = Crew(
    agents=[transaction_categorizer_agent, data_analyst_agent, financial_advisor_agent],
    tasks=[categorize_transactions, analyze_spending_trends, generate_financial_report],
    verbose=True,
)

In [ ]:
result = await finance_crew.kickoff_async(inputs={"file_path": FILE_PATH})
print(result.pydantic)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 001e6f3b-44ed-411b-acdf-c12b81d4eff8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the Transaction Loader tool to load the transactions CSV located at transactions.csv. Report the     │
│  total spend per category for every month present in the data, plus the overall total spend per month. Include  │
│  the month-over-month percent change the tool reports for each category.                                        │
│                                                                                                                 │
│  ID: 22c72d22-b8e2-4c4c-be78-f6b371a8144b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Transaction Categorization Specialist                                                            │
│                                                                                                                 │
│  Task: Use the Transaction Loader tool to load the transactions CSV located at transactions.csv. Report the     │
│  total spend per category for every month present in the data, plus the overall total spend per month. Include  │
│  the month-over-month percent change the tool reports for each category.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: transaction_loader                                                                                       │
│  Args: {'file_path': 'transactions.csv'}                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool transaction_loader executed with result: Monthly spend by category:

- Dining: 2026-04: $140.00, 2026-05: $200.00, 2026-06: $340.00
    change from 2026-04 to 2026-06: +142.9%
- Entertainment: 2026-04: $70.00, 2026-05: $90.00, 2026-06: $150....


╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: transaction_loader                                                                                       │
│  Output: Monthly spend by category:                                                                             │
│                                                                                                                 │
│  - Dining: 2026-04: $140.00, 2026-05: $200.00, 2026-06: $340.00                                                 │
│      change from 2026-04 to 2026-06: +142.9%                                                                    │
│  - Entertainment: 2026-04: $70.00, 2026-05: $90.00, 2026-06: $150.00                                            │
│      change from 2026-04 to 2026-06: +114.3%                                                                    │
│  - Groceries: 2026-04: $480.00, 2026-05: $510.00, 2026-06: $620.00                                              │
│      change from 2026-04 to 2026-06: +29.2%                                                                     │
│  - Healthcare: 2026-04: $40.00, 2026-05: $60.00, 2026-06: $45.00                                                │
│      change from 2026-04 to 2026-06: +12.5%                                                                     │
│  - Rent: 2026-04: $1,400.00, 2026-05: $1,400.00, 2026-06: $1,400.00                                             │
│      change from 2026-04 to 2026-06: +0.0%                                                                      │
│  - Shopping: 2026-04: $130.00, 2026-05: $180.00, 2026-06: $300.00                                               │
│      change from 2026-04 to 2026-06: +130.8%                                                                    │
│  - Subscriptions: 2026-04: $58.00, 2026-05: $62.00, 2026-06: $64.00                                             │
│      change from 2026-04 to 2026-06: +10.3%                                                                     │
│  - Transportation: 2026-04: $110.00, 2026-05: $125.00, 2026-06: $130.00                                         │
│      change from 2026-04 to 2026-06: +18.2%                                                                     │
│  - Travel: 2026-06: $450.00                                                                                     │
│  - Utilities: 2026-04: $195.00, 2026-05: $205.00, 2026-06: $210.00                                              │
│      change from 2026-04 to 2026-06: +7.7%                                                                      │
│                                                                                                                 │
│  Total spend by month:                                                                                          │
│  - 2026-04: $2,623.00                                                                                           │
│  - 2026-05: $2,832.00                                                                                           │
│  - 2026-06: $3,709.00                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (11 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Financial Health Advisor                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  summary='Your spending jumped 41% from April to June, driven by sharp rises in Dining, Shopping, and           │
│  Entertainment, pushing total expenses well above typical budget levels.' top_overspend_category='Dining'       │
│  health_status='critical' recommendations=['Cap Dining out to a maximum of $200 per month and track each meal   │
│  using a cash envelope or budgeting app', 'Trim Shopping spend by at least 50%—set a $150 monthly limit and     │
│  prioritize essential purchases', 'Cut Entertainment expenses in half by opting for free or low‑cost            │
│  activities and cancel any under‑used subscriptions']                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the categorized totals and the trend analysis, write a personal financial health report. Assign    │
│  an overall health_status of "healthy", "warning", or "critical" based on how many categories are trending up   │
│  sharply and how large those increases are relative to prior months. Provide between 2 and 4 specific,          │
│  prioritized recommendations tied directly to the categories driving the biggest overspend.                     │
│                                                                                                                 │
│  Agent: Personal Financial Health Advisor                                                                       │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 001e6f3b-44ed-411b-acdf-c12b81d4eff8                                                                       │
│  Final Output: {"summary":"Your spending jumped 41% from April to June, driven by sharp rises in Dining,        │
│  Shopping, and Entertainment, pushing total expenses well above typical budget                                  │
│  levels.","top_overspend_category":"Dining","health_status":"critical","recommendations":["Cap Dining out to a  │
│  maximum of $200 per month and track each meal using a cash envelope or budgeting app","Trim Shopping spend by  │
│  at least 50%—set a $150 monthly limit and prioritize essential purchases","Cut Entertainment expenses in half  │
│  by opting for free or low‑cost activities and cancel any under‑used subscriptions"]}                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

summary='Your spending jumped 41% from April to June, driven by sharp rises in Dining, Shopping, and Entertainment, pushing total expenses well above typical budget levels.' top_overspend_category='Dining' health_status='critical' recommendations=['Cap Dining out to a maximum of $200 per month and track each meal using a cash envelope or budgeting app', 'Trim Shopping spend by at least 50%—set a $150 monthly limit and prioritize essential purchases', 'Cut Entertainment expenses in half by opting for free or low‑cost activities and cancel any under‑used subscriptions']


Test

In [ ]:
import threading
import traceback

def run_test():
    try:
        finance_crew.test(n_iterations=1, eval_llm=llm)
    except Exception:
        traceback.print_exc()

t = threading.Thread(target=run_test)
t.start()
t.join()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a0375789-64a3-4a0c-ab5e-b580740a5c69                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🧪 Test Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Crew Test                                                                                             │
│  Crew: crew                                                                                                     │
│  ID: 001e6f3b-44ed-411b-acdf-c12b81d4eff8                                                                       │
│  Iterations: 1                                                                                                  │
│  Status: Running...                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the Transaction Loader tool to load the transactions CSV located at transactions.csv. Report the     │
│  total spend per category for every month present in the data, plus the overall total spend per month. Include  │
│  the month-over-month percent change the tool reports for each category.                                        │
│                                                                                                                 │
│  ID: fd6fd33d-90bc-4173-8680-ae82dec6a108                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Transaction Categorization Specialist                                                            │
│                                                                                                                 │
│  Task: Use the Transaction Loader tool to load the transactions CSV located at transactions.csv. Report the     │
│  total spend per category for every month present in the data, plus the overall total spend per month. Include  │
│  the month-over-month percent change the tool reports for each category.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: transaction_loader                                                                                       │
│  Args: {'file_path': 'transactions.csv'}                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool transaction_loader executed with result: Monthly spend by category:

- Dining: 2026-04: $140.00, 2026-05: $200.00, 2026-06: $340.00
    change from 2026-04 to 2026-06: +142.9%
- Entertainment: 2026-04: $70.00, 2026-05: $90.00, 2026-06: $150....

... (27 outputs left out)

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the categorized totals and the trend analysis, write a personal financial health report. Assign    │
│  an overall health_status of "healthy", "warning", or "critical" based on how many categories are trending up   │
│  sharply and how large those increases are relative to prior months. Provide between 2 and 4 specific,          │
│  prioritized recommendations tied directly to the categories driving the biggest overspend.                     │
│                                                                                                                 │
│  Agent: Personal Financial Health Advisor                                                                       │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                           Tasks Scores                                           
                                      (1-10 Higher is better)                                     
┏━━━━━━━━━━━━━━━━━━━━┯━━━━━━━┯━━━━━━━━━━━━┯━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┯━━┯━━┓
┃ Tasks/Crew/Agents  │ Run 1 │ Avg. Total │ Agents                                         │  │  ┃
┠────────────────────┼───────┼────────────┼────────────────────────────────────────────────┼──┼──┨
┃ Task 1             │  8.0  │    8.0     │ - Senior Transaction Categorization Specialist │  │  ┃
┃                    │       │            │                                                │  │  ┃
┃                    │       │            │                                                │  │  ┃
┃ Task 2             │  8.0  │    8.0     │ - Spending Trend Analyst                       │  │  ┃
┃                    │       │            │                                                │  │  ┃
┃                    │       │            │                                                │  │  ┃
┃ Task 3             │  9.0  │    9.0     │ - Personal Financial Health Advisor            │  │  ┃
┃                    │       │            │                                                │  │  ┃
┃ Crew               │ 8.33  │    8.3     │                                                │  │  ┃
┃ Execution Time (s) │  20   │     20     │                                                │  │  ┃
┗━━━━━━━━━━━━━━━━━━━━┷━━━━━━━┷━━━━━━━━━━━━┷━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┷━━┷━━┛

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a0375789-64a3-4a0c-ab5e-b580740a5c69                                                                       │
│  Final Output: {"summary":"Spending surged 41.5% from April to June, mainly due to sharp jumps in               │
│  discretionary categories, indicating your budget is slipping and savings are at                                │
│  risk.","top_overspend_category":"Dining","health_status":"warning","recommendations":["Cap monthly Dining      │
│  expenses at a realistic limit (e.g., $200) and track each outing against that budget","Trim Shopping spend by  │
│  30%: create a strict list, avoid impulse buys, and use cash-only purchases","Limit Entertainment to $100 per   │
│  month and prioritize low‑cost or free activities"]}                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

Train

In [ ]:
import threading
import traceback

def run_train():
    try:
        finance_crew.train(n_iterations=2, filename="finance_training.pkl")
    except Exception:
        traceback.print_exc()

t = threading.Thread(target=run_train)
t.start()
t.join()

╭─────────────────────────────────────────────── Training Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  📋 Crew Training Started                                                                                       │
│  Crew: crew                                                                                                     │
│  Time: 2026-08-26 14:19:13.239007+00:00                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b2b2c2da-59e0-49ff-8d4d-d8d3d868ebc2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the Transaction Loader tool to load the transactions CSV located at transactions.csv. Report the     │
│  total spend per category for every month present in the data, plus the overall total spend per month. Include  │
│  the month-over-month percent change the tool reports for each category.                                        │
│                                                                                                                 │
│  ID: 72a4b7bd-d107-4bcf-8428-733b43954a05                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Transaction Categorization Specialist                                                            │
│                                                                                                                 │
│  Task: Use the Transaction Loader tool to load the transactions CSV located at transactions.csv. Report the     │
│  total spend per category for every month present in the data, plus the overall total spend per month. Include  │
│  the month-over-month percent change the tool reports for each category.                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#9) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: transaction_loader                                                                                       │
│  Args: {'file_path': 'transactions.csv'}                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool transaction_loader executed with result: Monthly spend by category:

- Dining: 2026-04: $140.00, 2026-05: $200.00, 2026-06: $340.00
    change from 2026-04 to 2026-06: +142.9%
- Entertainment: 2026-04: $70.00, 2026-05: $90.00, 2026-06: $150....


... (76 outputs left out)

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Personal Financial Health Advisor                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  summary='Spending rose 26% in June, mainly due to dining (+143%), entertainment (+114%) and shopping (+131%).  │
│  Essentials stayed flat, but the sharp rise in discretionary categories signals a warning level of financial    │
│  health.' top_overspend_category='Dining' health_status='warning' recommendations=['Set a firm weekly dining    │
│  budget of $75 (≈$300/month) and track it in a budgeting app; treat any excess as a reset for the next week.',  │
│  'Create a weekly meal plan, shop with a strict list, and prep leftovers to reduce impulse restaurant           │
│  visits.', 'Cap monthly entertainment and shopping spend at $150 each and use cash envelopes or prepaid cards   │
│  to enforce the limits.']                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── 🎓 Training Feedback Required ─────────────────────────────────────────╮
│                                                                                                                 │
│  TRAINING MODE: Provide feedback to improve the agent's performance.                                            │
│                                                                                                                 │
│  This will be used to train better versions of the agent.                                                       │
│  Please provide detailed feedback about the result quality and reasoning process.                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

KeyboardInterrupt: 

Test (after)

In [ ]:

def run_test():
    try:
        finance_crew.test(n_iterations=1, eval_llm=llm)
    except Exception:
        traceback.print_exc()

t = threading.Thread(target=run_test)
t.start()
t.join()

the Flow

In [ ]:
from crewai.flow.flow import Flow, listen, router, start


class FinanceFlow(Flow):
    @start()
    def load_transactions(self):
        return {"file_path": FILE_PATH}

    @listen(load_transactions)
    def run_finance_crew(self, data):
        crew_result = finance_crew.kickoff(inputs=data)
        report = crew_result.pydantic
        self.state["report"] = report
        return report

    @router(run_finance_crew)
    def route_by_health(self, report):
        return report.health_status

    @listen("critical")
    def send_urgent_alert(self, _):
        report = self.state["report"]
        print("🚨 CRITICAL — urgent alert")
        print(f"Top overspend category: {report.top_overspend_category}")
        print(report.summary)
        for rec in report.recommendations:
            print(f"  - {rec}")
        return "urgent_alert_sent"

    @listen("warning")
    def schedule_review(self, _):
        report = self.state["report"]
        print("⚠️  WARNING — scheduling a budget review")
        print(report.summary)
        for rec in report.recommendations:
            print(f"  - {rec}")
        return "review_scheduled"

    @listen("healthy")
    def send_monthly_digest(self, _):
        report = self.state["report"]
        print("✅ HEALTHY — sending routine monthly digest")
        print(report.summary)
        return "digest_sent"

kick off


In [ ]:
flow = FinanceFlow()
flow.kickoff()

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: FinanceFlow                                                                                              │
│  ID: 70f9a2e4-5b0a-4947-a6fd-c34b44e2390e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: FinanceFlow                                                                                              │
│  ID: 70f9a2e4-5b0a-4947-a6fd-c34b44e2390e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_transactions                                                                                      │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: run_finance_crew                                                                                       │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: load_transactions                                                                                      │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d5480425-1026-41e2-b9e0-df5c7b44eca8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (23 outputs left out)

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: schedule_review                                                                                        │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: schedule_review                                                                                        │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: FinanceFlow                                                                                              │
│  ID: 70f9a2e4-5b0a-4947-a6fd-c34b44e2390e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

'review_scheduled'